# Single-home inference: Gibbs vs heuristic baseline

Runs Gibbs sampling on one EV home, derives $\hat C^{(n)}$ from the sampled $z$'s
(both directly and via the heuristic's logistic model), and produces:
- Hard and soft $z$ confusion matrices (ours vs heuristic)
- $C^{(n)}$ probability estimates (ours vs heuristic)
- Gibbs convergence diagnostics

In [8]:
import sys
from pathlib import Path

repo_root = str(Path('.').resolve().parents[1])
utils_dir = str(Path('.').resolve().parents[1] / 'notebooks' / 'utils')
for path in [repo_root, utils_dir]:
    if path not in sys.path:
        sys.path.insert(0, path)

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from models import graphical_model as gm
from models import first_diff_logistic as fdl
import convergence_plots as cvg

## Load data and fitted parameters

In [9]:
raw_train = pickle.load(open('../../data_processing/splits/train.pkl', 'rb'))

train_df = pd.concat(
    [df.assign(home_id=home_id, has_ev=has_ev, city=city)
     for home_id, (has_ev, city, df) in raw_train.items()]
).reset_index()
train_df.rename(columns={'car1': 'ev_load', 'load': 'total_load'}, inplace=True)
train_df['day'] = train_df['localminute'].dt.normalize()
train_df['time'] = train_df['localminute'].dt.strftime('%H:%M')
train_df = train_df.drop(columns='localminute')
train_df['time'] = pd.to_timedelta(train_df['time'] + ':00')
train_df['time_index'] = (train_df['time'].dt.total_seconds() // (15 * 60)).astype(int)
train_df['charge_state'] = train_df['charge_state'].astype('int')

full_day_mask = (
    train_df.groupby(['home_id', 'day'])['time_index']
    .transform(lambda x: x.nunique() == 96)
)
train_df = train_df[full_day_mask]
print(f'rows={len(train_df):,}  homes={train_df.home_id.nunique()}')

params = pickle.load(open('../../models/fitted_params.pkl', 'rb'))
print(params.summary())

rows=1,543,872  homes=54
ModelParams summary
----------------------------------------
EV States
  p_C                 = 0.1667
  pi_z                = [0.8304 0.0419 0.1276]
  P_z (rows sum to 1):
      off: [0.9875 0.0085 0.004 ]
      low: [0.3498 0.515  0.1352]
     high: [0.0232 0.1294 0.8474]

EV Charging Magnitudes
  Theta[ off]: mu=0.0000, sigma_Theta=0.0000, sigma^EV=0.0010
  Theta[ low]: mu=0.8727, sigma_Theta=0.1326, sigma^EV=0.4859
  Theta[high]: mu=3.5616, sigma_Theta=0.6219, sigma^EV=0.7378

Non-EV
  rho                 (||.||_2 = 1.0000, min=+0.060, max=+0.136, mean=+0.099)
  mu_alpha            = 9.6160
  sigma_alpha         = 5.9797
  sigma^NonEV_t       (min=0.589, median=1.007, max=1.268)


## Tune heuristic baseline (on full train set)

We need the fitted logistic model so we can later feed our Gibbs z-transition
counts into it and compare $\hat P(C=1)$ estimates.

In [10]:
homes_dict = gm.build_heuristic_homes(train_df)
logistic_model, w, lo, hi, md = fdl.tune(homes_dict)
heuristic_summary, heuristic_states = fdl.predict(logistic_model, homes_dict, w, lo, hi, md)
heuristic_summary['C_hat'] = (heuristic_summary['p_hat'] >= 0.5).astype(int)
print(heuristic_summary[heuristic_summary['has_ev'] == 1].to_string(index=False))

Best: window=6 (90 min), low=1.2, high=1.6, max_duration=16 (240 min)  (train avg precision=0.5502)
 dataid  has_ev  transitions_per_day  p_hat  C_hat
     27       1                2.372  0.805      1
    661       1                1.986  0.712      1
   3000       1                3.055  0.911      1
   4373       1                3.040  0.909      1
   4767       1                1.243  0.481      0
   6139       1                2.917  0.895      1
   7719       1                1.250  0.483      0
   8156       1                1.972  0.709      1
   9053       1                0.514  0.261      0


## Select one EV home

In [11]:
# ── CONFIGURE ──────────────────────────────────────────────────────────────
HOME_ID   = 27       # choose any EV home; EV home IDs: 27, 661, 3000, 4373,
                     #   4767, 6139, 7719, 8156, 9053
S_BURN    = 200      # burn-in iterations
S_RETAIN  = 500      # retained iterations
SEED      = 0
# ───────────────────────────────────────────────────────────────────────────

home_g   = train_df[train_df['home_id'] == HOME_ID].sort_values(['day', 'time_index'])
D        = home_g['day'].nunique()
home_x   = home_g['total_load'].to_numpy().reshape(D, 96).astype(float)
z_true   = home_g['charge_state'].to_numpy().reshape(D, 96)
true_c   = int(home_g['has_ev'].iloc[0])

heur_row      = heuristic_summary[heuristic_summary['dataid'] == HOME_ID].iloc[0]
heur_c_prob   = float(heur_row['p_hat'])
heur_z_states = heuristic_states[HOME_ID].reshape(D, 96)  # heuristic per-timestep states

print(f'Home {HOME_ID}:  true C={true_c},  D={D} days')
print(f'Heuristic: p_hat={heur_c_prob:.3f},  '
      f'transitions/day={heur_row["transitions_per_day"]:.2f}')

Home 27:  true C=1,  D=183 days
Heuristic: p_hat=0.805,  transitions/day=2.37


## Run Gibbs (always with C=1; derive C from sampled z's)

We always pass `C_hat=1` and let the model determine $P(C=1)$ from the z samples:
- **Direct estimate**: fraction of samples where any $z_{d,t} \neq \mathtt{off}$
- **Via heuristic logistic**: feed per-sample transition counts into the pre-trained logistic model

In [12]:
rng = np.random.default_rng(SEED)
inference = gm.infer_home(
    home_x, params, C_hat=1,
    S_burn=S_BURN, S=S_RETAIN,
    rng=rng, home_id=HOME_ID,
    verbose=True, record_traces=True,
)

  [home 27] C_hat=1, D=183 → running Gibbs (200 burn-in + 500 retained)
    iter    1/700 [burn-in]  α=7.451  Θ_low=1.006  Θ_high=3.017  logL=-17305.8  (0.0s)
    iter    2/700 [burn-in]  α=7.018  Θ_low=1.062  Θ_high=3.175  logL=-17133.4  (0.0s)
    iter    3/700 [burn-in]  α=7.107  Θ_low=1.008  Θ_high=3.151  logL=-17102.5  (0.1s)
    iter  100/700 [burn-in]  α=7.086  Θ_low=1.083  Θ_high=3.234  logL=-17124.5  (1.9s)
    iter  200/700 [burn-in]  α=6.835  Θ_low=1.025  Θ_high=3.196  logL=-17102.3  (3.7s)
    iter  201/700 [keep  ]  α=6.944  Θ_low=1.079  Θ_high=3.270  logL=-17085.2  (3.7s)
    iter  300/700 [keep  ]  α=6.915  Θ_low=1.094  Θ_high=3.215  logL=-17101.7  (5.6s)  Δα/σ=167.2%
    iter  400/700 [keep  ]  α=6.972  Θ_low=1.059  Θ_high=3.198  logL=-17099.0  (7.5s)  Δα/σ=29.7%
    iter  500/700 [keep  ]  α=6.936  Θ_low=1.039  Θ_high=3.216  logL=-17151.1  (9.3s)  Δα/σ=168.1%
    iter  600/700 [keep  ]  α=7.075  Θ_low=1.245  Θ_high=3.294  logL=-17110.6  (11.2s)  Δα/σ=137.4%
    iter  7

TypeError: HomeInference.__init__() got an unexpected keyword argument 'c_from_z_samples'

## Convergence diagnostics

In [ ]:
figs = cvg.plot_all_diagnostics(inference)
for fig in figs:
    plt.show()

## Evaluation

### Helpers

In [ ]:
K = 3
STATE_LABELS = ['off', 'low', 'high']

def hard_z_confusion(z_true, z_pred, normalize_rows=True):
    """Standard confusion matrix.  Row = true state, col = predicted state."""
    cm = np.zeros((K, K), dtype=float)
    for t, p in zip(z_true.ravel(), z_pred.ravel()):
        cm[int(t), int(p)] += 1
    if normalize_rows:
        row_sums = cm.sum(axis=1, keepdims=True)
        cm = np.where(row_sums > 0, cm / row_sums, 0.0)
    return cm

def soft_z_confusion(z_true, z_marginals, normalize_rows=True):
    """Expected confusion matrix: adds posterior P(z=k) instead of a hard count.
    Row = true state, col = predicted state.
    """
    cm = np.zeros((K, K), dtype=float)
    for k_true in range(K):
        mask = (z_true == k_true)              # (D, T)
        cm[k_true] = z_marginals[mask].sum(axis=0)
    if normalize_rows:
        row_sums = cm.sum(axis=1, keepdims=True)
        cm = np.where(row_sums > 0, cm / row_sums, 0.0)
    return cm

def print_confusion(cm, title, labels=STATE_LABELS):
    print(f'\n{title}')
    n = len(labels)
    header = '        ' + '  '.join(f'{l:>8}' for l in labels)
    print(header)
    for i in range(n):
        row_str = '  '.join(f'{cm[i,j]:>8.3f}' if isinstance(cm[i,j], float)
                            else f'{cm[i,j]:>8d}' for j in range(n))
        print(f'{labels[i]:>6}  {row_str}')

### z confusion matrices

In [ ]:
# Gibbs: hard prediction (MAP z)
gibbs_hard_cm = hard_z_confusion(z_true, inference.z_hat)
print_confusion(gibbs_hard_cm,
    f'Gibbs hard z confusion (home {HOME_ID}, row=true, col=pred, row-normalised)')

# Gibbs: soft prediction (posterior marginals)
gibbs_soft_cm = soft_z_confusion(z_true, inference.z_marginals)
print_confusion(gibbs_soft_cm,
    'Gibbs soft z confusion (expected confusion under posterior, row-normalised)')

# Heuristic: hard prediction
heur_hard_cm = hard_z_confusion(z_true, heur_z_states[:D, :])
print_confusion(heur_hard_cm,
    'Heuristic hard z confusion (state machine on total load, row-normalised)')

### C probability estimates

In [ ]:
# Our method 1: fraction of samples with any non-off z
c_prob_direct = float(inference.c_from_z_samples.mean())

# Our method 2: feed z transition counts into heuristic logistic
# NOTE: logistic was trained on heuristic state-machine counts, not Gibbs z;
#       may be miscalibrated for non-EV homes but should be fine here.
c_prob_via_heuristic = gm.c_prob_from_z_via_heuristic(inference, logistic_model)

print(f'True C                           = {true_c}')
print(f'Gibbs P(C=1) — direct from z     = {c_prob_direct:.4f}  '
      f'→ C_hat = {int(c_prob_direct >= 0.5)}')
print(f'Gibbs P(C=1) — via logistic on z = {c_prob_via_heuristic:.4f}  '
      f'→ C_hat = {int(c_prob_via_heuristic >= 0.5)}')
print(f'Heuristic P(C=1) — on raw load   = {heur_c_prob:.4f}  '
      f'→ C_hat = {int(heur_c_prob >= 0.5)}')

print()
print(f'Mean Gibbs z-transitions/day = {inference.z_transitions_per_day_samples.mean():.2f}')
print(f'Heuristic transitions/day    = {heur_row["transitions_per_day"]:.2f}')
print()
print('(Both transition-rate estimates should agree for a clear EV home.  '
      'Disagreement signals calibration issues.)')

## Posterior z heatmap

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

axes[0].imshow(z_true.T, aspect='auto', cmap='viridis', vmin=0, vmax=2,
               interpolation='nearest')
axes[0].set_title(f'Home {HOME_ID}: ground-truth z')
axes[0].set_ylabel('time')

axes[1].imshow(inference.z_hat.T, aspect='auto', cmap='viridis', vmin=0, vmax=2,
               interpolation='nearest')
axes[1].set_title('Gibbs MAP z̃ (hard)')
axes[1].set_ylabel('time')

p_charging = 1.0 - inference.z_marginals[:, :, 0]
im = axes[2].imshow(p_charging.T, aspect='auto', cmap='Reds', vmin=0, vmax=1,
                    interpolation='nearest')
axes[2].set_title('Posterior P(z ≠ off)')
axes[2].set_ylabel('time')
plt.colorbar(im, ax=axes[2], fraction=0.02)

axes[3].imshow(heur_z_states[:D].T, aspect='auto', cmap='viridis', vmin=0, vmax=2,
               interpolation='nearest')
axes[3].set_title('Heuristic state (state machine on total load)')
axes[3].set_xlabel('day')
axes[3].set_ylabel('time')

plt.tight_layout()
plt.show()